# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Next, let's display all record sets along with their `@id`s and list the fields for each.


In [ ]:
# List record sets and their IDs
record_sets = dataset.metadata.record_sets
print("Record Sets in the Dataset:")
for rs in record_sets:
    print(f"- {rs['@id']} ({rs.get('name', 'Unnamed')})")
    if 'fields' in rs:
        print("  Fields:")
        for field in rs['fields']:
            print(f"    - {field['@id']} ({field.get('name', 'Unnamed')})")
        print()

# Also print columns in each record set
for rs in record_sets:
    if 'columns' in rs:
        print(f"Columns for {rs['@id']}:")
        for col in rs['columns']:
            print(f"  - {col['@id']} ({col.get('name', 'Unnamed')})")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We'll extract the main survey records for analysis (using their `@id`).

In [ ]:
# We'll now load the records for each record set

# Collect all record set IDs
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

# Inspect column names of the main record set
if len(dataframes) > 0:
    # Just show info for the first record set
    main_rs_id = record_set_ids[0]
    print(f"Columns for record set {main_rs_id}:")
    print(dataframes[main_rs_id].columns.tolist())
    dataframes[main_rs_id].head()
else:
    print("No records loaded. Please check that the record set IDs are correct and accessible.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, and grouping data by key attributes.

We'll demonstrate filtering for GAD-7 scores (>10), normalizing, and grouping by `gender`.

In [ ]:
# Choose the main record set and fields/columns by their @ids
main_record_set_id = record_set_ids[0]
df = dataframes[main_record_set_id].copy()

# Let's identify numeric columns (e.g., GAD-7, PHQ-9, PSQ total scores)
numeric_field_id = None
possible_numeric_fields = ['gad7_score', 'phq9_score', 'psq_score']
for field in possible_numeric_fields:
    if field in df.columns:
        numeric_field_id = field
        break
if numeric_field_id is None:
    numeric_field_id = df.select_dtypes('number').columns[0]
print(f"Using numeric field: {numeric_field_id}")

# Filter records (e.g., GAD-7 > 10)
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = \
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a key demographic field (e.g., 'gender')
group_field = 'gender'
if group_field in df.columns:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())
else:
    print(f"Field {group_field} not found in columns. Available columns: {df.columns.tolist()}")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of GAD-7 scores and visualize mean scores by gender.

In [ ]:
# Distribution of GAD-7 scores
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id], kde=True, bins=15, color='skyblue')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Mean GAD-7 scores by gender
if group_field in df.columns:
    plt.figure(figsize=(6, 4))
    sns.barplot(x=group_field, y=numeric_field_id, data=df, estimator='mean')
    plt.title(f"Mean {numeric_field_id} by {group_field}")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Kilifi Mental Health Survey dataset enables demographic and clinical analysis.
- We demonstrated loading the dataset, reviewing structure by record set and field `@id`, and extracting survey records.
- EDA steps revealed distributions of GAD-7 scores and differences by gender.
- The dataset is suitable for further statistical analysis and model development on mental health indicators by leveraging structured Croissant metadata.
